# Fuzzy (trigram) vs edit-distance retrieval quality, with embeddings on top

Follow-up to `malayalam_fuzzy_matching_research.ipynb`'s threshold analysis, but testing
the real question directly: when a query is misspelled, what does the **actual retrieval
layer** (`fuzzy_search()` in `linguaalayam/database/queries.py`) return today versus what
an edit-distance version of the same query would return — the full candidate lists, not a
single hand-picked pair — and does adding a semantic-embedding check on top of either
candidate list help surface the right answer more reliably.

Runs against the real local dev Postgres (`docker compose up -d db`), mirroring
`fuzzy_search()`'s and `similarity_search()`'s exact SQL so results reflect production,
not a simplified stand-in.

In [1]:
import os
import re

import numpy as np
import pandas as pd
import psycopg2
from dotenv import dotenv_values
from sentence_transformers import SentenceTransformer

env = dotenv_values("../.env")
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname=env.get("DB_NAME", "linguaalayam"),
    user=env.get("DB_USER", "postgres"),
    password=env["DB_PASSWORD"],
)
cur = conn.cursor()

# fuzzystrmatch isn't in any migration (confirmed: only pg_trgm and vector are) —
# it's always been enabled ad hoc in a notebook session, same here.
cur.execute("CREATE EXTENSION IF NOT EXISTS fuzzystrmatch;")
conn.commit()

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Connected. Model loaded.")

/home/sachin_nandakumar/projects/LinguAalayam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected. Model loaded.


## 1. The two retrieval functions, mirroring production exactly

`trigram_search` is `fuzzy_search()` from `database/queries.py` verbatim (same `similarity()`
call, same default `threshold=0.3`, same `limit`/ordering) — just issued as raw SQL instead
of through SQLAlchemy, so it can run standalone in this notebook.

`levenshtein_search` is the edit-distance equivalent that doesn't exist as app code yet
(confirmed: no migration adds `fuzzystrmatch`, no function uses `levenshtein()`) — same
shape, so the two are directly comparable. Score is normalized to `1 - lev/max_len`, the
exact metric the earlier threshold-research notebook settled on.

In [2]:
def trigram_search(query: str, threshold: float = 0.3, limit: int = 10) -> pd.DataFrame:
    """Verbatim SQL mirror of fuzzy_search() in database/queries.py."""
    cur.execute(
        """
        SELECT headword, source, embed_text, similarity(headword, %s) AS score
        FROM dictionary_entries
        WHERE similarity(headword, %s) > %s
        ORDER BY score DESC
        LIMIT %s;
        """,
        (query, query, threshold, limit),
    )
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=["headword", "source", "embed_text", "score"])


def levenshtein_search(query: str, threshold: float = 0.3, limit: int = 10) -> pd.DataFrame:
    """Same shape as trigram_search, but scored by normalized edit distance.

    1 - levenshtein(headword, query) / greatest(len(headword), len(query), 1) — the metric
    the threshold-research notebook (section 6) recommended switching to.
    """
    cur.execute(
        """
        SELECT headword, source, embed_text,
               1.0 - levenshtein(headword, %s)::float
                     / GREATEST(char_length(headword), char_length(%s), 1) AS score
        FROM dictionary_entries
        WHERE 1.0 - levenshtein(headword, %s)::float
                     / GREATEST(char_length(headword), char_length(%s), 1) > %s
        ORDER BY score DESC
        LIMIT %s;
        """,
        (query, query, query, query, threshold, limit),
    )
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=["headword", "source", "embed_text", "score"])


print("Defined.")

Defined.


## 2. Test queries

Reusing known cases from the earlier notebook (a Malayalam typo, an English typo), plus
two new ones to broaden coverage: a longer Malayalam word (should favor trigram less
harshly, per the earlier length-dependence finding) and a case with a real nearby
collision risk.

In [3]:
test_queries = [
    ("കുരലാരം", "കൂരലാരം", "ML short typo — the original bug report"),
    ("garaent", "garment", "EN short typo"),
    ("ദംശിക്കുക", "ദർശിക്കുക", "ML — real collision risk nearby (bite vs see)"),
    ("എഴുതുന്നു", "എഴുതുക", "ML longer word, inflected form"),
]
pd.DataFrame(test_queries, columns=["query", "intended/nearby", "why"])

,query,intended/nearby,why
0,കുരലാരം,കൂരലാരം,ML short typo — the original bug report
1,garaent,garment,EN short typo
2,ദംശിക്കുക,ദർശിക്കുക,ML — real collision risk nearby (bite vs see)
3,എഴുതുന്നു,എഴുതുക,"ML longer word, inflected form"


## 3. Candidate lists, side by side

For each query: what trigram returns, what edit-distance returns, and where they diverge
— does the intended word even appear in each list, and at what rank.

In [4]:
def candidate_report(query: str, intended: str) -> pd.DataFrame:
    trig = trigram_search(query).assign(method="trigram")
    lev = levenshtein_search(query).assign(method="levenshtein")
    combined = pd.concat([trig, lev], ignore_index=True)
    combined["query"] = query
    combined["is_intended"] = combined["headword"] == intended
    return combined[["query", "method", "headword", "source", "score", "is_intended"]]


all_candidates = pd.concat(
    [candidate_report(q, intended) for q, intended, _ in test_queries], ignore_index=True
)

for q, intended, why in test_queries:
    sub = all_candidates[all_candidates["query"] == q].sort_values(
        ["method", "score"], ascending=[True, False]
    )
    print(f"\n=== {q!r} -> intended {intended!r} ({why}) ===")
    for method in ("trigram", "levenshtein"):
        m = sub[sub["method"] == method]
        found = m["is_intended"].any()
        rank = (m["is_intended"].values.argmax() + 1) if found else None
        print(f"  {method:11s}: {len(m)} candidates, intended found={found}"
              + (f" at rank {rank}" if found else ""))
    display(sub.drop(columns="query"))


=== 'കുരലാരം' -> intended 'കൂരലാരം' (ML short typo — the original bug report) ===
  trigram    : 10 candidates, intended found=True at rank 1
  levenshtein: 10 candidates, intended found=True at rank 1


,method,headword,source,score,is_intended
10,levenshtein,കൂരലാരം,datuk,0.857143,True
11,levenshtein,കൊരലാരം,datuk,0.857143,False
12,levenshtein,കുരലം,sayahna,0.714286,False
13,levenshtein,കുലഹരം,datuk,0.714286,False
14,levenshtein,കുഠാരം,datuk,0.714286,False
15,levenshtein,കുമാരം,sayahna,0.714286,False
16,levenshtein,കരവാരം,datuk,0.714286,False
17,levenshtein,കുഠാരം,sayahna,0.714286,False
18,levenshtein,കുരരം,datuk,0.714286,False
19,levenshtein,കുരലം,datuk,0.714286,False



=== 'garaent' -> intended 'garment' (EN short typo) ===
  trigram    : 8 candidates, intended found=True at rank 2
  levenshtein: 10 candidates, intended found=True at rank 1


,method,headword,source,score,is_intended
28,levenshtein,garment,ekkurup,0.857143,True
29,levenshtein,garden,ekkurup,0.714286,False
30,levenshtein,garret,ekkurup,0.714286,False
31,levenshtein,ardent,ekkurup,0.714286,False
32,levenshtein,Garment,olam_enml,0.714286,False
33,levenshtein,Parent,olam_enml,0.714286,False
34,levenshtein,grant,ekkurup,0.714286,False
35,levenshtein,parent,ekkurup,0.714286,False
36,levenshtein,Sacrament,olam_enml,0.666667,False
37,levenshtein,Seargent,olam_enml,0.625000,False



=== 'ദംശിക്കുക' -> intended 'ദർശിക്കുക' (ML — real collision risk nearby (bite vs see)) ===
  trigram    : 10 candidates, intended found=True at rank 4
  levenshtein: 10 candidates, intended found=True at rank 3


,method,headword,source,score,is_intended
48,levenshtein,ദംശിക്കുക,datuk,1.000000,False
49,levenshtein,അംശിക്കുക,datuk,0.888889,False
50,levenshtein,ദർശിക്കുക,datuk,0.888889,True
51,levenshtein,ആദേശിക്കുക,datuk,0.800000,False
52,levenshtein,സംശയിക്കുക,datuk,0.800000,False
53,levenshtein,നശിക്കുക,datuk,0.777778,False
54,levenshtein,കർശിക്കുക,datuk,0.777778,False
55,levenshtein,ദാഹിക്കുക,datuk,0.777778,False
56,levenshtein,ദുഷിക്കുക,datuk,0.777778,False
57,levenshtein,അശിക്കുക,datuk,0.777778,False



=== 'എഴുതുന്നു' -> intended 'എഴുതുക' (ML longer word, inflected form) ===
  trigram    : 10 candidates, intended found=True at rank 4
  levenshtein: 10 candidates, intended found=False


,method,headword,source,score,is_intended
68,levenshtein,വഴുതുന്നു,sayahna,0.888889,False
69,levenshtein,എഴുന്നു,datuk,0.777778,False
70,levenshtein,വഴുകുന്നു,sayahna,0.777778,False
71,levenshtein,എഴുനിന്നു,sayahna,0.777778,False
72,levenshtein,തഴുകുന്നു,sayahna,0.777778,False
73,levenshtein,മെഴുകുന്നു,sayahna,0.700000,False
74,levenshtein,മുഴുകുന്നു,sayahna,0.700000,False
75,levenshtein,പരുകുന്നു,sayahna,0.666667,False
76,levenshtein,മരുളുന്നു,sayahna,0.666667,False
77,levenshtein,മരുവുന്നു,sayahna,0.666667,False


## 4. Does string-similarity rank even matter once embeddings are added?

For every candidate either method returned, embed the query and compare against the
candidate's **own stored embedding** (production's actual `similarity_search()` column,
not a re-embedding of `embed_text` here — this is the real vector already in the table) via
cosine similarity, mirroring `similarity_search()`'s math (`1 - cosine_distance`) exactly.
Then check: within each method's candidate list, does re-sorting by semantic similarity
instead of string score change which candidate ends up on top — and does that help or hurt
relative to the intended word?

In [5]:
def fetch_embedding(headword: str, source: str) -> np.ndarray | None:
    cur.execute(
        "SELECT embedding FROM dictionary_entries WHERE headword = %s AND source = %s LIMIT 1;",
        (headword, source),
    )
    row = cur.fetchone()
    if row is None or row[0] is None:
        return None
    # pgvector returns the column as a "[0.1,0.2,...]" string over plain psycopg2 —
    # confirmed the hard way in the other notebook: np.array() on that string alone
    # silently produces a unicode-dtype array, not floats.
    import ast

    return np.array(ast.literal_eval(row[0]), dtype=float)


def add_semantic_scores(df: pd.DataFrame, query: str) -> pd.DataFrame:
    query_emb = model.encode(query, normalize_embeddings=True)
    sims = []
    for _, row in df.iterrows():
        cand_emb = fetch_embedding(row["headword"], row["source"])
        if cand_emb is None:
            sims.append(None)
            continue
        cand_emb = cand_emb / np.linalg.norm(cand_emb)
        sims.append(float(np.dot(query_emb, cand_emb)))
    return df.assign(semantic_score=sims)


reranked_summary = []
for q, intended, why in test_queries:
    sub = all_candidates[all_candidates["query"] == q]
    for method in ("trigram", "levenshtein"):
        m = add_semantic_scores(sub[sub["method"] == method].copy(), q)
        if m.empty:
            continue
        top_by_string = m.sort_values("score", ascending=False).iloc[0]
        top_by_semantic = (
            m.dropna(subset=["semantic_score"]).sort_values("semantic_score", ascending=False)
        )
        top_by_semantic = top_by_semantic.iloc[0] if not top_by_semantic.empty else None
        reranked_summary.append({
            "query": q,
            "intended": intended,
            "method": method,
            "top_by_string_score": top_by_string["headword"],
            "top_by_string_correct": top_by_string["headword"] == intended,
            "top_by_semantic_score": top_by_semantic["headword"] if top_by_semantic is not None else None,
            "top_by_semantic_correct": (top_by_semantic["headword"] == intended) if top_by_semantic is not None else None,
        })

pd.DataFrame(reranked_summary)

,query,intended,method,top_by_string_score,top_by_string_correct,top_by_semantic_score,top_by_semantic_correct
0,കുരലാരം,കൂരലാരം,trigram,കൂരലാരം,True,കൂരലാരം,True
1,കുരലാരം,കൂരലാരം,levenshtein,കൂരലാരം,True,കൂരലാരം,True
2,garaent,garment,trigram,Garment,False,Garb,False
3,garaent,garment,levenshtein,garment,True,garret,False
4,ദംശിക്കുക,ദർശിക്കുക,trigram,ദംശിക്കുക,False,ദംശിക്കുക,False
5,ദംശിക്കുക,ദർശിക്കുക,levenshtein,ദംശിക്കുക,False,ദംശിക്കുക,False
6,എഴുതുന്നു,എഴുതുക,trigram,എഴുന്നു,False,എഴുതുക,True
7,എഴുതുന്നു,എഴുതുക,levenshtein,വഴുതുന്നു,False,എഴുന്നു,False


## 5. Read this table before drawing conclusions

If `top_by_semantic_correct` is `True` more often than `top_by_string_correct` — especially
in rows where the string-score ranking got it wrong — that's a real, usable signal:
re-ranking (not replacing) each method's candidate list by semantic similarity would help.
If the two columns mostly agree, or semantic re-ranking is wrong exactly where string
scoring was already right, embeddings aren't adding anything beyond what the string
metric already provides for these cases.

This directly extends section 13's single-pair test in the other notebook, but on the
**real candidate sets** either algorithm would actually return — not a hand-picked pair —
so this result should be trusted over that one if they disagree.

## 6. Logging this comparison to MLflow

The existing eval harness (`linguaalayam/eval/`) is built around a fixed labeled query set
scored via hit@k/MRR against the merged production pipeline — it has no notion of "trigram
vs Levenshtein" as parallel alternatives, so this doesn't fit `run_eval()` directly. But its
MLflow logging *shape* (`_log_retrieval_to_mlflow` in `eval/main.py`) transfers cleanly:
tags + params + metrics + a CSV artifact. Logging this run the same way, so it's not lost
and is comparable later if this becomes a recurring check.

In [6]:
import io

import mlflow

mlflow.set_experiment("retrieval_pipeline")

with mlflow.start_run(run_name="trigram_vs_levenshtein_vs_embedding"):
    mlflow.set_tags({"comparison": "trigram_vs_levenshtein", "stage": "exploratory"})
    mlflow.log_params({
        "n_queries": len(test_queries),
        "embedding_model": "paraphrase-multilingual-mpnet-base-v2",
        "trigram_threshold": 0.3,
        "levenshtein_threshold": 0.3,
    })

    summary_df = pd.DataFrame(reranked_summary)
    for method in ("trigram", "levenshtein"):
        m = summary_df[summary_df["method"] == method]
        mlflow.log_metric(f"{method}_string_top1_accuracy", m["top_by_string_correct"].mean())
        mlflow.log_metric(f"{method}_semantic_top1_accuracy", m["top_by_semantic_correct"].mean())

    buf = io.StringIO()
    summary_df.to_csv(buf, index=False)
    mlflow.log_text(buf.getvalue(), "comparison_summary.csv")

    buf2 = io.StringIO()
    all_candidates.to_csv(buf2, index=False)
    mlflow.log_text(buf2.getvalue(), "all_candidates.csv")

print("Logged. Run `mlflow ui --port 5001` from the repo root to inspect.")

2026/10/12 13:52:37 INFO mlflow.tracking.fluent: Experiment with name 'retrieval_pipeline' does not exist. Creating a new experiment.


Logged. Run `mlflow ui --port 5001` from the repo root to inspect.


## Summary

- Built the real candidate lists both retrieval algorithms would actually return for each
  test query (not single hand-picked pairs), mirroring `fuzzy_search()`'s exact SQL for
  trigram, and a parallel `levenshtein()`-based query for edit distance.
- Added a semantic-similarity re-ranking layer on top of each method's candidates, using
  each entry's **real stored embedding** (not a re-encoding of `embed_text`).
- See section 5's table for whether semantic re-ranking actually beats string-score ranking
  on these cases — read that table before concluding either way.
- Logged as an MLflow run (`retrieval_pipeline` experiment, tagged `exploratory`) so this
  is comparable if repeated later with more queries.

**Not built here, flagged for later** (per discussion, not part of this notebook):
- A local UI to interactively try arbitrary queries against both methods — useful once the
  underlying metric/re-ranking question above is settled, not before.
- Whether this should become a recurring, non-exploratory part of the MLflow-tracked eval
  harness (`linguaalayam/eval/`) rather than a one-off notebook run — worth revisiting once
  there's a real query set sized enough for statistically meaningful hit@k, not just 4
  hand-picked cases.

In [7]:
cur.close()
conn.close()